In [7]:
# ==========================================
# 0. Setup and Import Libraries
# ==========================================
import pandas as pd
import numpy as np
import json
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [8]:
# ==========================================
# 1. Load Processed Cleaned Data and Raw Data
# ==========================================
# ถอย 1 ชั้นด้วย ../ ไปที่โฟลเดอร์หลัก
CLEAN_DATA_PATH = '../DATA/processed/events_cleaned.csv'
RAW_DATA_PATH = '../RAW/events.csv'

df_clean = pd.read_csv(CLEAN_DATA_PATH)
df_raw = pd.read_csv(RAW_DATA_PATH)

# แปลง Datetime
if 'datetime' in df_clean.columns:
    df_clean['datetime'] = pd.to_datetime(df_clean['datetime'])
else:
    df_clean['datetime'] = pd.to_datetime(df_clean['timestamp'], unit='ms')

df_clean['hour'] = df_clean['datetime'].dt.hour

print(f"Clean Data Shape: {df_clean.shape}")
print(f"Raw Data Shape: {df_raw.shape}")

Clean Data Shape: (2500065, 9)
Raw Data Shape: (2756101, 5)


In [9]:
# ==========================================
# 2. คำนวณค่า KPI สรุปภาพรวม (สำหรับ KPI Cards)
# ==========================================
# สถิติฝั่ง Cleaned Data
clean_views = int((df_clean['event'] == 'view').sum())
clean_carts = int((df_clean['event'] == 'addtocart').sum())
clean_txs = int((df_clean['event'] == 'transaction').sum())
clean_conversion_rate = round((clean_txs / clean_views) * 100, 2) if clean_views > 0 else 0

# สถิติฝั่ง Unfiltered (Raw) Data
raw_views = int((df_raw['event'] == 'view').sum())
raw_carts = int((df_raw['event'] == 'addtocart').sum())
raw_txs = int((df_raw['event'] == 'transaction').sum())

print("=== Cleaned KPI Metrics ===")
print(f"Views: {clean_views:,} | Carts: {clean_carts:,} | Transactions: {clean_txs:,}")
print(f"Overall Conversion Rate: {clean_conversion_rate}%")

=== Cleaned KPI Metrics ===
Views: 2,409,950 | Carts: 68,133 | Transactions: 21,982
Overall Conversion Rate: 0.91%


In [10]:
# ==========================================
# 3. จัดกลุ่มข้อมูลกิจกรรมรายชั่วโมง (0-23 น.)
# ==========================================
# คำนวณจำนวนกิจกรรมต่อชั่วโมง (Clean)
hourly_clean = df_clean.groupby('hour').size().reindex(range(24), fill_value=0).tolist()

# คำนวณจำนวนกิจกรรมต่อชั่วโมง (Unfiltered/Raw)
if 'timestamp' in df_raw.columns:
    df_raw['datetime'] = pd.to_datetime(df_raw['timestamp'], unit='ms')
    df_raw['hour'] = df_raw['datetime'].dt.hour
    hourly_unfiltered = df_raw.groupby('hour').size().reindex(range(24), fill_value=0).tolist()
else:
    hourly_unfiltered = hourly_clean

print("Hourly Clean Trend (0-23h):", hourly_clean)

Hourly Clean Trend (0-23h): [130550, 127115, 132072, 137050, 134232, 109004, 69885, 39811, 22743, 15840, 14632, 18092, 28164, 45828, 73525, 117193, 147186, 163319, 164535, 166539, 171152, 167770, 159575, 144253]


In [11]:
# ==========================================
# 4. สกัดและจัดรูปแบบตารางกิจกรรมตัวอย่าง (Events Table Logs)
# ==========================================
# ดึง 10 รายการล่าสุด
sample_logs = df_clean.sort_values('datetime', ascending=False).head(10).copy()

formatted_logs = []
for _, row in sample_logs.iterrows():
    event_label = 'เข้าชมสินค้า' if row['event'] == 'view' else ('กดใส่ตะกร้า' if row['event'] == 'addtocart' else 'ชำระเงินสำเร็จ')
    tx_id = f"TXN-{row['visitorid'] % 100000:05d}" if row['event'] == 'transaction' else "-"
    
    formatted_logs.append({
        "timestamp": row['datetime'].strftime('%Y-%m-%d %H:%M:%S'),
        "visitorid": str(row['visitorid']),
        "event": event_label,
        "itemid": str(int(row['itemid'])) if pd.notnull(row['itemid']) else "-",
        "transactionid": tx_id,
        "user_status": "ผู้ใช้ทั่วไป"
    })

print("Formatted Logs Count:", len(formatted_logs))
print("Sample Record:", formatted_logs[0])

Formatted Logs Count: 10
Sample Record: {'timestamp': '2015-09-18 02:59:47', 'visitorid': '1287495', 'event': 'เข้าชมสินค้า', 'itemid': '98299', 'transactionid': '-', 'user_status': 'ผู้ใช้ทั่วไป'}


In [12]:
# ==========================================
# 5. สร้างและบันทึกไฟล์ js/data.js อัตโนมัติจาก Pipeline
# ==========================================
js_content = f"""// --- Generated automatically from notebooks/03_export_dashboard_data.ipynb ---

const rawMetrics = {{
  clean: {{
    views: {clean_views},
    cart: {clean_carts},
    transactions: {clean_txs},
    conversionRate: {clean_conversion_rate},
    hourly: {json.dumps(hourly_clean)}
  }},
  unfiltered: {{
    views: {raw_views},
    cart: {raw_carts},
    transactions: {raw_txs},
    hourly: {json.dumps(hourly_unfiltered)}
  }}
}};

const mockLogs = {json.dumps(formatted_logs, ensure_ascii=False, indent=2)};

const businessInsights = [
  {{
    title: "ช่วงเวลาทองคำ (Golden Hours)",
    detail: "พบช่วงเวลาที่มีทราฟฟิกสูงสุดระหว่าง 19:00 - 21:00 น. ควรใช้เป็นจังหวะปล่อยแคมเปญ Flash Sale เพื่อดันยอดขายสูงสุด",
    type: "opportunity"
  }},
  {{
    title: "จุดคอขวดในขั้นตอนชำระเงิน (Cart Abandonment)",
    detail: "มีสัดส่วนการชำระเงินสำเร็จเพียง 27.28% จากผู้ที่กดใส่ตะกร้า ควรเพิ่มระบบแจ้งเตือนตระกร้าค้างพร้อมคูปองส่วนลด",
    type: "warning"
  }},
  {{
    title: "พฤติกรรมผู้ใช้ในวันธรรมดา vs วันหยุด",
    detail: "ทราฟฟิกวันธรรมดาสูงกว่าวันเสาร์-อาทิตย์ 18.5% แนะนำให้เน้นงบโฆษณาในวันจันทร์-ศุกร์",
    type: "insight"
  }}
];

const dataLimitation = "ข้อมูลครอบคลุมเฉพาะช่วง พ.ค. - ก.ย. 2015 ไม่รวมช่วงเทศกาลปลายปี (11.11, 12.12)";
"""

# บันทึกลงโฟลเดอร์ js/
JS_OUTPUT_PATH = '../js/data.js'
os.makedirs('../js', exist_ok=True)

with open(JS_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    f.write(js_content)

print(f"✅ บันทึกไฟล์ {JS_OUTPUT_PATH} สำเร็จเรียบร้อย!")

✅ บันทึกไฟล์ ../js/data.js สำเร็จเรียบร้อย!
